<a href="https://colab.research.google.com/github/Shizukem/cu-i-k-/blob/main/b%C3%A0i_6.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import files

uploaded = files.upload()

Saving vietnam_regions_2024.csv to vietnam_regions_2024.csv


In [2]:
import pandas as pd
import numpy as np

In [3]:
# Load the dataset
df_regions = pd.read_csv('vietnam_regions_2024.csv')

# Print actual columns to debug KeyError
# print("DataFrame columns:", df_regions.columns.tolist())

# Define REGIONS_S (Region Names)
REGIONS_S = df_regions['region_name_en'].tolist()

# Define the criteria columns for X_raw (Decision Matrix)
criteria_columns = [
    'population_million',
    'grdp_trillion_VND',
    'grdp_growth_pct',
    'grdp_per_capita_million_VND', # Corrected name
    'ai_readiness_0_100',          # Replaces AI investment/research
    'digital_index_0_100',         # Replaces innovation_index
    'rd_intensity_pct',            # Added for R&D focus
    'trained_labor_pct',           # Replaces education_level_index
    'gini_coef',                   # Proxy for unemployment/poverty (lower is better)
    'internet_penetration_pct'     # General digital infrastructure
]
X_raw = df_regions[criteria_columns].values

# Define IS_BENEFIT (True if higher is better, False if lower is better for each criterion)
IS_BENEFIT = np.array([
    True,  # population_million
    True,  # grdp_trillion_VND
    True,  # grdp_growth_pct
    True,  # grdp_per_capita_million_VND
    True,  # ai_readiness_0_100
    True,  # digital_index_0_100
    True,  # rd_intensity_pct
    True,  # trained_labor_pct
    False, # gini_coef (lower is better)
    True   # internet_penetration_pct
])

# Define W_EXPERT (Expert Weights)
# These are placeholder equal weights; you can adjust them based on expert opinion.
W_EXPERT = np.array([1/len(criteria_columns)] * len(criteria_columns))

print("Variables `REGIONS_S`, `X_raw`, `IS_BENEFIT`, and `W_EXPERT` have been defined.")
print("REGIONS_S:", REGIONS_S)
print("IS_BENEFIT:", IS_BENEFIT)
print("W_EXPERT (initial equal weights):", W_EXPERT)

Variables `REGIONS_S`, `X_raw`, `IS_BENEFIT`, and `W_EXPERT` have been defined.
REGIONS_S: ['Northern Midlands and Mountains', 'Red River Delta', 'North Central and South Central Coast', 'Central Highlands', 'Southeast', 'Mekong Delta']
IS_BENEFIT: [ True  True  True  True  True  True  True  True False  True]
W_EXPERT (initial equal weights): [0.1 0.1 0.1 0.1 0.1 0.1 0.1 0.1 0.1 0.1]


In [4]:
def topsis(X, w, is_benefit):
    """
    TOPSIS chuẩn 5 bước theo đề bài.
    X          : ma trận quyết định (n_alt × n_crit)
    w          : vector trọng số (n_crit,), tổng = 1
    is_benefit : list bool — True = lợi ích, False = chi phí
    Trả về: C_star (n_alt,), S_star (n_alt,), S_neg (n_alt,)
    """
    X     = np.array(X, dtype=float)
    w     = np.array(w, dtype=float)
    n, m  = X.shape

    # Bước 1: Chuẩn hóa vector
    # r_ij = x_ij / sqrt(Σ x_ij²)
    denom = np.sqrt((X ** 2).sum(axis=0))
    denom[denom == 0] = 1e-12
    R = X / denom

    # Bước 2: Ma trận chuẩn hóa có trọng số
    V = R * w

    # Bước 3: Xác định A+ và A-
    A_pos = np.where(is_benefit, V.max(axis=0), V.min(axis=0))
    A_neg = np.where(is_benefit, V.min(axis=0), V.max(axis=0))

    # Bước 4: Khoảng cách Euclide đến A+ và A-
    S_pos = np.sqrt(((V - A_pos) ** 2).sum(axis=1))
    S_neg = np.sqrt(((V - A_neg) ** 2).sum(axis=1))

    # Bước 5: Hệ số gần gũi tương đối C*
    C_star = S_neg / (S_pos + S_neg)

    return C_star, S_pos, S_neg, V, A_pos, A_neg, R

In [5]:
def entropy_weights(X):
    """
    Tính trọng số khách quan bằng phương pháp Entropy.
    - Chuẩn hóa P_ij = x_ij / Σ_i(x_ij)
    - Entropy E_j = -k * Σ_i(P_ij * ln(P_ij))  với k = 1/ln(n)
    - Độ phân tán d_j = 1 - E_j
    - Trọng số w_j = d_j / Σ d_j
    """
    X  = np.array(X, dtype=float)
    n  = X.shape[0]
    k  = 1.0 / np.log(n)

    # Chuẩn hóa tỷ lệ
    col_sum = X.sum(axis=0)
    col_sum[col_sum == 0] = 1e-12
    P = X / col_sum

    # Entropy từng tiêu chí
    with np.errstate(divide='ignore', invalid='ignore'):
        lnP = np.where(P > 0, np.log(P), 0.0)
    E = -k * (P * lnP).sum(axis=0)

    # Độ phân tán & trọng số
    d = 1 - E
    w = d / d.sum()
    return w

In [6]:
def ahp_weights_from_data(X, is_benefit):
    """
    AHP đơn giản: xây dựng ma trận so sánh cặp từ dữ liệu thực.
    Tỷ số a_ij = x_i / x_j (với tiêu chí lợi ích)
                 x_j / x_i (với tiêu chí chi phí)
    Eigenvector chính → trọng số AHP.
    Trả về trọng số chuẩn hóa.
    """
    X = np.array(X, dtype=float)
    n = X.shape[0]
    weights_ahp = np.zeros(n)

    for i in range(n):
        for j in range(n):
            if i == j:
                weights_ahp[i] += 1.0
            else:
                total = 0.0
                for col in range(X.shape[1]):
                    xi = X[i, col]
                    xj = X[j, col]
                    if xi > 0 and xj > 0:
                        if is_benefit[col]:
                            total += xi / xj
                        else:
                            total += xj / xi
                    else:
                        total += 1.0
                weights_ahp[i] += total / X.shape[1]

    weights_ahp /= weights_ahp.sum()
    return weights_ahp


def ahp_topsis_combined(X, w_expert, is_benefit):
    """
    Kết hợp AHP (trọng số tiêu chí) + TOPSIS (xếp hạng phương án).
    Ở đây dùng trọng số chuyên gia cho tiêu chí (như đề bài),
    AHP tính điểm tổng hợp cho phương án từ ma trận so sánh cặp.
    """
    X = np.array(X, dtype=float)
    n, m = X.shape

    # Chuẩn hóa min-max để đồng nhất thang đo
    X_norm = np.zeros_like(X)
    for j in range(m):
        mn, mx = X[:, j].min(), X[:, j].max()
        if mx > mn:
            if is_benefit[j]:
                X_norm[:, j] = (X[:, j] - mn) / (mx - mn)
            else:                                      # chi phí: đảo
                X_norm[:, j] = (mx - X[:, j]) / (mx - mn)
        else:
            X_norm[:, j] = 1.0

    # Điểm AHP = tổng trọng số × giá trị chuẩn hóa (SAW weighted sum)
    scores = X_norm @ w_expert
    scores_norm = scores / scores.sum()
    return scores_norm, X_norm

In [7]:
print('=' * 68)
print('  BÀI 6 — TOPSIS XẾP HẠNG 6 VÙNG KINH TẾ THEO ƯU TIÊN ĐẦU TƯ AI')
print('=' * 68)

print('\n[ Câu 6.4.1 ] TOPSIS với trọng số chuyên gia')
print(f'  w = {W_EXPERT.tolist()}')
print('-' * 60)

C1, S1p, S1n, V1, A1p, A1n, R1 = topsis(X_raw, W_EXPERT, IS_BENEFIT)

# Xếp hạng
rank1 = C1.argsort()[::-1] + 1
order1 = C1.argsort()[::-1]

df1 = pd.DataFrame({
    'Vùng':    REGIONS_S,
    'S+':      S1p,
    'S-':      S1n,
    'C*':      C1,
    'Xếp hạng': rank1,
}).set_index('Vùng')

print('\n  Kết quả TOPSIS (trọng số chuyên gia):')
print(df1.to_string(float_format=lambda v: f'{v:.4f}'))
print(f'\n  Top-3 vùng ưu tiên đầu tư AI:')
for i, idx in enumerate(order1[:3]):
    print(f'    #{i+1}: {REGIONS_S[idx]:<12}  C* = {C1[idx]:.4f}')

  BÀI 6 — TOPSIS XẾP HẠNG 6 VÙNG KINH TẾ THEO ƯU TIÊN ĐẦU TƯ AI

[ Câu 6.4.1 ] TOPSIS với trọng số chuyên gia
  w = [0.1, 0.1, 0.1, 0.1, 0.1, 0.1, 0.1, 0.1, 0.1, 0.1]
------------------------------------------------------------

  Kết quả TOPSIS (trọng số chuyên gia):
                                          S+     S-     C*  Xếp hạng
Vùng                                                                
Northern Midlands and Mountains       0.1077 0.0237 0.1805         2
Red River Delta                       0.0113 0.1161 0.9117         5
North Central and South Central Coast 0.0742 0.0555 0.4279         3
Central Highlands                     0.1204 0.0052 0.0418         6
Southeast                             0.0163 0.1119 0.8728         1
Mekong Delta                          0.0939 0.0375 0.2855         4

  Top-3 vùng ưu tiên đầu tư AI:
    #1: Red River Delta  C* = 0.9117
    #2: Southeast     C* = 0.8728
    #3: North Central and South Central Coast  C* = 0.4279


In [8]:
print('\n[ Câu 6.4.2 ] Trọng số khách quan Entropy')
print('-' * 60)

# Với Gini (chi phí), đảo ngược trước khi tính entropy
X_for_entropy = X_raw.copy()

# Define M as the number of criteria (columns in X_raw)
M = X_raw.shape[1]

for j in range(M):
    if not IS_BENEFIT[j]:
        X_for_entropy[:, j] = 1.0 / (X_raw[:, j] + 1e-12)

W_ENT = entropy_weights(X_for_entropy)

# Assuming CRITERIA_S is defined (list of column names from df_regions)
CRITERIA_S = criteria_columns # Assuming criteria_columns holds the names

print('  Trọng số Entropy:')
for j, (cs, w) in enumerate(zip(CRITERIA_S, W_ENT)):
    print(f'    {cs:<18}: w = {w:.4f}')
print(f'  Tổng: {W_ENT.sum():.6f}')

C2, S2p, S2n, V2, A2p, A2n, R2 = topsis(X_raw, W_ENT, IS_BENEFIT)
rank2 = C2.argsort()[::-1] + 1
order2 = C2.argsort()[::-1]

df2 = pd.DataFrame({
    'Vùng':    REGIONS_S,
    'S+':      S2p,
    'S-':      S2n,
    'C*':      C2,
    'Xếp hạng': rank2,
}).set_index('Vùng')

print('\n  Kết quả TOPSIS (trọng số Entropy):')
print(df2.to_string(float_format=lambda v: f'{v:.4f}'))
print(f'\n  Top-3 vùng (Entropy):')
for i, idx in enumerate(order2[:3]):
    print(f'    #{i+1}: {REGIONS_S[idx]:<12}  C* = {C2[idx]:.4f}')

# So sánh xếp hạng
print('\n  So sánh xếp hạng Chuyên gia vs. Entropy:')
print(f'  {"Vùng":<12} {"Rank CE":>8} {"Rank EN":>8} {"ΔRank":>8}')
print('  ' + '-' * 42)
for i, vung in enumerate(REGIONS_S):
    delta = int(rank1[i]) - int(rank2[i])
    mk = ' ← thay đổi lớn' if abs(delta) >= 2 else ''
    print(f'  {vung:<12} {rank1[i]:>8} {rank2[i]:>8} {delta:>+8}{mk}')


[ Câu 6.4.2 ] Trọng số khách quan Entropy
------------------------------------------------------------
  Trọng số Entropy:
    population_million: w = 0.0782
    grdp_trillion_VND : w = 0.2396
    grdp_growth_pct   : w = 0.0030
    grdp_per_capita_million_VND: w = 0.0914
    ai_readiness_0_100: w = 0.1615
    digital_index_0_100: w = 0.0694
    rd_intensity_pct  : w = 0.2741
    trained_labor_pct : w = 0.0729
    gini_coef         : w = 0.0014
    internet_penetration_pct: w = 0.0085
  Tổng: 1.000000

  Kết quả TOPSIS (trọng số Entropy):
                                          S+     S-     C*  Xếp hạng
Vùng                                                                
Northern Midlands and Mountains       0.2130 0.0250 0.1052         2
Red River Delta                       0.0117 0.2281 0.9511         5
North Central and South Central Coast 0.1530 0.0859 0.3595         3
Central Highlands                     0.2316 0.0043 0.0184         6
Southeast                             0.0

In [9]:
print('\n[ Câu 6.4.3 ] Phân tích độ nhạy w_AI từ 0.10 → 0.40')
print('-' * 60)

# j=3 là AI Readiness (index 3)
w_AI_range = np.arange(0.10, 0.41, 0.05)

# Define N as the number of alternatives (regions)
N = len(REGIONS_S)

sens_C      = np.zeros((len(w_AI_range), N))
sens_rank   = np.zeros((len(w_AI_range), N), dtype=int)

# Trọng số gốc không kể AI Readiness
w_others_base = W_EXPERT.copy()
w_others_base[3] = 0.0  # bỏ AI
w_sum_others = w_others_base.sum()  # tổng trọng số còn lại

for k, w_ai in enumerate(w_AI_range):
    # Chuẩn hóa lại: tăng/giảm tỷ lệ các tiêu chí khác
    scale = (1.0 - w_ai) / w_sum_others
    w_new = w_others_base * scale
    w_new[3] = w_ai
    assert abs(w_new.sum() - 1.0) < 1e-9

    Ck, _, _, _, _, _, _ = topsis(X_raw, w_new, IS_BENEFIT)
    sens_C[k]    = Ck
    sens_rank[k] = Ck.argsort()[::-1] + 1

print(f'  {"w_AI":>8}  ' + '  '.join(f'{v:>8}' for v in REGIONS_S))
print('  ' + '-' * 70)
for k, w_ai in enumerate(w_AI_range):
    row = '  '.join(f'{sens_rank[k, i]:>8}' for i in range(N))
    print(f'  {w_ai:>8.2f}  {row}')

# Top-3 ổn định không?
top3_stable = True

# Get the indices of the actual top 3 ranked regions for the baseline
baseline_order_indices = sens_C[0].argsort()[::-1]
baseline_top3_regions_indices = set(baseline_order_indices[:3])

for k in range(1, len(w_AI_range)):
    current_order_indices = sens_C[k].argsort()[::-1]
    current_top3_regions_indices = set(current_order_indices[:3])
    if current_top3_regions_indices != baseline_top3_regions_indices:
        top3_stable = False
        break

print(f'\n  Top-3 có ổn định khi w_AI thay đổi? '
      f'{"✓ Có" if top3_stable else "✗ Không — có sự thay đổi"}')


[ Câu 6.4.3 ] Phân tích độ nhạy w_AI từ 0.10 → 0.40
------------------------------------------------------------
      w_AI  Northern Midlands and Mountains  Red River Delta  North Central and South Central Coast  Central Highlands  Southeast  Mekong Delta
  ----------------------------------------------------------------------
      0.10         2         5         3         6         1         4
      0.15         2         5         3         6         1         4
      0.20         2         5         3         6         1         4
      0.25         2         5         3         6         1         4
      0.30         2         5         3         6         1         4
      0.35         5         2         3         6         1         4
      0.40         5         2         3         6         4         1

  Top-3 có ổn định khi w_AI thay đổi? ✓ Có


In [10]:
print('\n[ Câu 6.4.4 ] So sánh TOPSIS vs. AHP đơn giản')
print('-' * 60)

ahp_scores, X_norm_ahp = ahp_topsis_combined(X_raw, W_EXPERT, IS_BENEFIT)
rank_ahp = ahp_scores.argsort()[::-1] + 1
order_ahp = ahp_scores.argsort()[::-1]

df_ahp = pd.DataFrame({
    'Vùng':       REGIONS_S,
    'AHP score':  ahp_scores,
    'AHP rank':   rank_ahp,
    'TOPSIS C*':  C1,
    'TOPSIS rank':rank1,
}).set_index('Vùng')

print('\n  So sánh AHP vs. TOPSIS (trọng số chuyên gia):')
print(df_ahp.to_string(float_format=lambda v: f'{v:.4f}'))

print(f'\n  Top-3 AHP:')
for i in range(3):
    print(f'    #{i+1}: {REGIONS_S[order_ahp[i]]:<12}  score = {ahp_scores[order_ahp[i]]:.4f}')

print(f'\n  Top-3 TOPSIS:')
for i in range(3):
    print(f'    #{i+1}: {REGIONS_S[order1[i]]:<12}  C* = {C1[order1[i]]:.4f}')

# Đồng thuận top-3
top3_topsis = set(order1[:3])
top3_ahp    = set(order_ahp[:3])
agree = len(top3_topsis & top3_ahp)
print(f'\n  Số vùng cùng nằm trong top-3 (AHP ∩ TOPSIS): {agree}/3')


[ Câu 6.4.4 ] So sánh TOPSIS vs. AHP đơn giản
------------------------------------------------------------

  So sánh AHP vs. TOPSIS (trọng số chuyên gia):
                                       AHP score  AHP rank  TOPSIS C*  TOPSIS rank
Vùng                                                                              
Northern Midlands and Mountains           0.0834         2     0.1805            2
Red River Delta                           0.3307         5     0.9117            5
North Central and South Central Coast     0.1622         3     0.4279            3
Central Highlands                         0.0140         6     0.0418            6
Southeast                                 0.3055         1     0.8728            1
Mekong Delta                              0.1042         4     0.2855            4

  Top-3 AHP:
    #1: Red River Delta  score = 0.3307
    #2: Southeast     score = 0.3055
    #3: North Central and South Central Coast  score = 0.1622

  Top-3 TOPSIS:
    #1: R